## Stage 1: Install dependencies and setting up GPU environment

In [1]:
!pip install tensorflow

## Stage 2: Importing project dependencies

pip install tensorflow

In [2]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence

## Stage 3: Dataset preprocessing

### Setting up dataset parameters

In [3]:
number_of_words = 20000
max_len = 100

### Loading the IMDB dataset

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=number_of_words)

### Padding all sequences to be the same length

In [4]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=number_of_words)

In [5]:
X_train = tf.keras.preprocessing.sequence.pad_sequences(X_train, maxlen=max_len)

In [6]:
X_test = tf.keras.preprocessing.sequence.pad_sequences(X_test, maxlen=max_len)

### Setting up Embedding Layer parameters

In [7]:
vocab_size = number_of_words
vocab_size

20000

In [8]:
embed_size = 128

## Step 4: Building a Recurrent Neural Network

### Defining the model

In [9]:
model = tf.keras.Sequential()

### Adding the Embeding Layer

In [10]:
model.add(tf.keras.layers.Embedding(vocab_size, embed_size, input_shape=(X_train.shape[1],)))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Adding the LSTM Layer

- units: 128
- activation: tanh

In [11]:
model.add(tf.keras.layers.LSTM(units=128, activation='tanh'))

### Adding the Dense output layer

- units: 1
- activation: sigmoid

In [12]:
model.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

### Compiling the model

In [13]:
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,691,713 (10.27 MB)

 Trainable params: 2,691,713 (10.27 MB)

 Non-trainable params: 0 (0.00 B)

### Training the model

In [15]:
model.fit(X_train, y_train, epochs=30, batch_size=64)

Epoch 1/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.7448 - loss: 0.5036
Epoch 2/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.8580 - loss: 0.3415
Epoch 3/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.8831 - loss: 0.2862
Epoch 4/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9040 - loss: 0.2477
Epoch 5/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9172 - loss: 0.2147
Epoch 6/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9352 - loss: 0.1769
Epoch 7/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9474 - loss: 0.1476
Epoch 8/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9566 - loss: 0.1229
Epoch 9/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9665 - loss: 0.0991
Epoch 10/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9724 - loss: 0.0799
Epoch 11/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9782 - loss: 0.0661
Epoch 12/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms

### Evaluating the model

In [16]:
test_loss, test_acurracy = model.evaluate(X_test, y_test)

782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8348 - loss: 1.2343


In [17]:
print("Test accuracy: {}".format(test_acurracy))

Test accuracy: 0.8347600102424622


In [18]:
# Define the input text for prediction
input_text = "I can't recommend this movie enough! The performances were Oscar-worthy, the script was intelligent, and the message resonated deeply. It left me with a sense of joy and appreciation for the art of storytelling."

# Tokenize and preprocess the input text
word_to_index = imdb.get_word_index()
input_sequence = [word_to_index.get(word.lower(), 0) + 3 for word in input_text.split()]
input_data = sequence.pad_sequences([input_sequence], maxlen=max_len)

# Make the prediction
predicted_sentiment = model.predict(input_data)[0][0]

# Convert the prediction to a human-readable sentiment
sentiment_label = 'Positive' if predicted_sentiment > 0.5 else 'Negative'

# Print the predicted sentiment
print("Predicted Sentiment:", sentiment_label)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step
Predicted Sentiment: Positive


In [19]:
import numpy as np
from tensorflow.keras.preprocessing import sequence

def predict_sentiment(input_text):
    # Tokenize and preprocess the input text
    word_to_index = imdb.get_word_index()
    input_sequence = [word_to_index.get(word.lower(), 0) + 3 for word in input_text.split()]
    input_data = sequence.pad_sequences([input_sequence], maxlen=max_len)

    # Make the prediction
    predicted_sentiment = model.predict(input_data)[0][0]

    # Convert the prediction to a human-readable sentiment
    sentiment_label = 'Positive' if predicted_sentiment > 0.5 else 'Negative'

    return sentiment_label

In [20]:
predict_sentiment("Absolute waste of time! The plot was predictable, the acting was wooden, and the special effects were laughable. Avoid at all costs!")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step


'Negative'

In [21]:
!pip install anvil-uplink==0.5.2

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 4.5 MB/s eta 0:00:00
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)
  Attempting uninstall: anvil-uplink
    Found existing installation: anvil-uplink 0.7.0
    Uninstalling anvil-uplink-0.7.0:
      Successfully uninstalled anvil-uplink-0.7.0


In [22]:
import anvil.server

anvil.server.connect("server_ED63CDT76HB7GNNWQD7Q2OOK-CVVW3KR3PGS5GWMW")

Connecting to wss://anvil.works/uplink
Anvil websocket open
Connected to "Default Environment" as SERVER


In [23]:
import anvil.media

@anvil.server.callable
def predict_sentiment(input_text):
    # Tokenize and preprocess the input text
    word_to_index = imdb.get_word_index()
    input_sequence = [word_to_index.get(word.lower(), 0) + 3 for word in input_text.split()]
    input_data = sequence.pad_sequences([input_sequence], maxlen=max_len)

    # Make the prediction
    predicted_sentiment = model.predict(input_data)[0][0]

    # Convert the prediction to a human-readable sentiment
    sentiment_label = 'Positive' if predicted_sentiment > 0.5 else 'Negative'

    return sentiment_label

In [ ]:
anvil.server.wait_forever()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
